In [ ]:
# ==============================================================================
# PROJECT: Smart Parking Availability Predictor Using Machine Learning
# Course: B.Tech Computer Science & Engineering — Semester V Machine Learning Lab
# Practical Experiment Reference: Exp 1 (EDA), Exp 5 (Trees), Exp 9 (8 Classifiers)
# Academic Year: 2026–27 | 15-Mark Mini Project Lab Practical
# ==============================================================================


# Smart Parking Availability Predictor Using Machine Learning

**Student Name:** [Your Name] | **Roll No:** [Your Roll No] | **Batch:** B1  
**Department:** Computer Science and Engineering  
**Course:** Machine Learning Laboratory (Semester V)  

---

### Project Overview & 15-Mark Evaluation Rubric:
1. **Part 1: Data Cleaning & Splitting (3 Marks)** — Memory-efficient parsing, delimiter detection, filtering 0-capacity rows, same-segment lag engineering, leak-free target construction, and chronological train-val-test splitting.
2. **Part 2: State-of-the-Art Models Comparison & Performance Analytics (4 Marks)** — Implementing all 8 syllabus classification algorithms + Persistence Baseline, macro F1 evaluation, confusion matrix error analysis, and hyperparameter tuning.
3. **Part 3: UI Deployment & Cross-Platform Accessibility (4 Marks)** — Building an interactive Streamlit dashboard with time-slot forecasting, smart-city color coding (Available: Green, Nearly Full: Amber, Full: Red), and CSV export.
4. **Part 4: Code Quality, GitHub Repo, Report Writing & Viva (4 Marks)** — Modular student-level code, clear comments for lab evaluation, complete academic report, and 20+ viva questions.


## Cell 1: Install and Import Required Libraries
In this cell, we import standard data science and machine learning packages.
Notice we follow the clean student coding style from our lab experiments (`pandas`, `numpy`, `matplotlib`, `seaborn`, and `scikit-learn`).


In [ ]:
# Install Streamlit and tunneling client for Colab deployment (uncomment if running in Colab)
# !pip install -q streamlit pyngrok joblib

import os
import sys
import math
import joblib
import datetime
from collections import Counter

# Core scientific computing libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing & model selection
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# 8 Syllabus Classifiers (Experiment 9)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("All required libraries imported successfully!")
print("Python Version:", sys.version.split()[0])
print("Pandas Version:", pd.__version__)


## Cell 2: Upload and Inspect Dataset
In this cell, we handle the dataset upload and inspect its physical structure.
The SFpark dataset (`data.csv`) is ~332 MB with 5,080,320 rows and uses a **semicolon (`;`) delimiter**.
To avoid memory overflow in Colab, we detect the delimiter and load only the 4 core documented columns:
`timestamp`, `segmentid`, `capacity`, and `occupied`.


In [ ]:
# Option A: Upload from local machine via Colab files upload
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = "data.csv"

# Delimiter detection function
def detect_delimiter(file_path):
    if not os.path.exists(file_path):
        return ";" # Default SFpark delimiter
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
        if ";" in first_line:
            return ";"
        elif "," in first_line:
            return ","
        elif "\t" in first_line:
            return "\t"
    return ";"

delim = detect_delimiter(DATA_PATH)
print(f"Detected Delimiter: '{delim}'")

# Memory-optimized data types for the 4 core columns
CORE_COLUMNS = ["timestamp", "segmentid", "capacity", "occupied"]
DTYPES = {
    "segmentid": "int32",
    "capacity": "int16",
    "occupied": "int16"
}

# If running on actual data.csv:
if os.path.exists(DATA_PATH):
    file_size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f"File found! Size: {file_size_mb:.2f} MB")
    
    # Read first 10,000 rows preview or full data in chunks
    df_raw = pd.read_csv(
        DATA_PATH,
        sep=delim,
        usecols=CORE_COLUMNS,
        dtype=DTYPES,
        nrows=500000 # Read sample or full chunks
    )
else:
    print("Notice: 'data.csv' not found locally. Synthesizing exact SFpark report distribution for demonstration...")
    # Generate realistic SFpark structure matching report (420 segments, 5-min intervals)
    dates = pd.date_range("2013-06-13 00:04", periods=2000, freq="5min")
    records = []
    np.random.seed(42)
    for seg_id in range(101, 151): # 50 sample segments for quick demonstration
        cap = int(np.random.choice([6, 8, 10, 12, 16], p=[0.2, 0.4, 0.2, 0.1, 0.1]))
        base_occ = np.random.uniform(0.2, 0.8)
        for dt in dates:
            # Diurnal hourly effect
            hour_effect = math.sin((dt.hour - 6) / 24 * 2 * math.pi) * 0.25
            occ_rate = np.clip(base_occ + hour_effect + np.random.normal(0, 0.08), 0.0, 1.0)
            occupied = int(round(occ_rate * cap))
            records.append({"timestamp": str(dt), "segmentid": seg_id, "capacity": cap, "occupied": occupied})
    df_raw = pd.DataFrame(records)

print(f"\nDataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("\nFirst 5 Rows:")
display(df_raw.head()) if 'display' in globals() else print(df_raw.head())

print("\nMissing Values Count:")
print(df_raw.isnull().sum())

print("\nExact Duplicates Count:", df_raw.duplicated().sum())
print("Duplicate (timestamp, segmentid) Count:", df_raw.duplicated(subset=["timestamp", "segmentid"]).sum())


## Cell 3: Exploratory Data Analysis (EDA) — Experiment 1 Syllabus
In this cell, we explore:
1. Capacity and Occupancy distributions.
2. Occupancy Percentage distribution.
3. Diurnal (hourly) and day-of-week trends.
4. Availability 3-Class distribution:
   - **Available:** `[0%, 50%)`
   - **Nearly Full:** `[50%, 80%)`
   - **Full:** `[80%, 100%]`
5. Outlier Analysis: Why 100% occupancy and capacities > 22.5 are legitimate conditions, NOT errors.


In [ ]:
# Parse timestamp to datetime
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

# Filter zero-capacity records before calculating percentage (to prevent division by zero)
df_valid = df_raw[df_raw['capacity'] > 0].copy()
zero_cap_count = len(df_raw) - len(df_valid)
print(f"Zero capacity records removed: {zero_cap_count:,} ({(zero_cap_count/len(df_raw))*100:.1f}%)")

# Calculate Occupancy Percentage
df_valid['occupancy_pct'] = (df_valid['occupied'] / df_valid['capacity']) * 100.0

# Extract time features
df_valid['hour'] = df_valid['timestamp'].dt.hour
df_valid['dayofweek'] = df_valid['timestamp'].dt.dayofweek
df_valid['weekday_name'] = df_valid['timestamp'].dt.day_name()

# Define project class labels
def get_availability_class(pct):
    if pct < 50.0:
        return "Available"
    elif pct < 80.0:
        return "Nearly Full"
    else:
        return "Full"

df_valid['current_class'] = df_valid['occupancy_pct'].apply(get_availability_class)

# Display Class Distribution
print("\n--- Availability Class Distribution ---")
class_dist = df_valid['current_class'].value_counts(normalize=True) * 100
for cls_name, pct in class_dist.items():
    print(f"{cls_name:12s}: {pct:.2f}%")

# Visualizations: 4-Panel EDA Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Capacity Distribution
sns.histplot(df_valid['capacity'], discrete=True, color='#3b82f6', ax=axes[0, 0])
axes[0, 0].set_title("1. Distribution of Parking Segment Capacity", fontweight='bold')
axes[0, 0].set_xlabel("Capacity (bays)")
axes[0, 0].set_ylabel("Count")

# 2. Hourly Average Occupancy Percentage
hourly_avg = df_valid.groupby('hour')['occupancy_pct'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, marker='o', color='#8b5cf6', linewidth=2.5)
axes[0, 1].set_title("2. Average Parking Occupancy by Hour of Day", fontweight='bold')
axes[0, 1].set_xlabel("Hour of Day (0 to 23)")
axes[0, 1].set_ylabel("Average Occupancy (%)")
axes[0, 1].set_xticks(range(0, 24, 2))

# 3. Class Distribution Countplot
colors = {'Available': '#10b981', 'Nearly Full': '#f59e0b', 'Full': '#ef4444'}
sns.countplot(data=df_valid, x='current_class', order=['Available', 'Nearly Full', 'Full'], palette=colors, ax=axes[1, 0])
axes[1, 0].set_title("3. Parking Availability Class Balance", fontweight='bold')
axes[1, 0].set_xlabel("Availability Class")
axes[1, 0].set_ylabel("Frequency")

# 4. Occupancy Percentage Boxplot across Days of the Week
order_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=df_valid, x='weekday_name', y='occupancy_pct', order=order_days, palette='Blues', ax=axes[1, 1])
axes[1, 1].set_title("4. Occupancy Distribution by Day of Week", fontweight='bold')
axes[1, 1].set_xlabel("Day of Week")
axes[1, 1].set_ylabel("Occupancy (%)")
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Cell 4: Data Cleaning and Preprocessing
In this cell, we implement our clean preprocessing steps:
1. Handle missing values (confirmed 0 in core columns).
2. Handle duplicates (confirmed 0 exact duplicates).
3. Validate logical boundaries: `0 <= occupied <= capacity` and `capacity > 0`.
4. Cast variables into memory-efficient representations (`int16`, `float32`).
5. Document why zero-capacity records are dropped (roads under maintenance or zero regulated spaces where division by zero would corrupt the target).


In [ ]:
print("--- Data Cleaning Summary ---")
# 1. Verify capacity positive
assert (df_valid['capacity'] > 0).all(), "Error: Zero or negative capacity detected!"

# 2. Verify occupied valid
assert (df_valid['occupied'] >= 0).all(), "Error: Negative occupied detected!"
assert (df_valid['occupied'] <= df_valid['capacity']).all(), "Error: Occupancy exceeds capacity!"

# 3. Optimize types
df_clean = df_valid.copy()
df_clean['segmentid'] = df_clean['segmentid'].astype('int32')
df_clean['capacity'] = df_clean['capacity'].astype('int16')
df_clean['occupied'] = df_clean['occupied'].astype('int16')
df_clean['occupancy_pct'] = df_clean['occupancy_pct'].astype('float32')

print("All physical integrity checks passed:")
print(f"- Minimum Capacity: {df_clean['capacity'].min()}, Maximum: {df_clean['capacity'].max()}")
print(f"- Minimum Occupied: {df_clean['occupied'].min()}, Maximum: {df_clean['occupied'].max()}")
print(f"- Total Clean Records: {len(df_clean):,}")


## Cell 5: Feature Engineering and Target Alignment
In this cell, we create the predictive features and the 60-minute future target:
1. **Time Features:** `hour`, `dayofweek`, `is_weekend` (1 if Saturday/Sunday, else 0).
2. **Same-Segment Historical Lag Features:**
   - Readings are taken every 5 minutes.
   - `occ_lag_15`: Occupancy percentage 15 mins prior (shift +3 readings).
   - `occ_lag_30`: Occupancy percentage 30 mins prior (shift +6 readings).
   - `occ_lag_60`: Occupancy percentage 60 mins prior (shift +12 readings).
3. **Future Prediction Target:**
   - `future_occ_pct`: Occupancy percentage 60 minutes into the future (shift -12 readings).
   - Discretized into 3 classes: `0 = Available`, `1 = Nearly Full`, `2 = Full`.

**Critical Anti-Leakage Rule:** Lags and future targets are grouped strictly by `segmentid` and sorted chronologically. Rows without complete historical lags or future targets are dropped. The future target is **NEVER** included in input features `X`.


In [ ]:
# Sort chronologically within each parking segment
df_clean = df_clean.sort_values(by=['segmentid', 'timestamp']).reset_index(drop=True)

# Generate lags and future target within each segment group
def create_lags_and_target(group):
    # Readings are every 5 mins: 3 steps = 15m, 6 steps = 30m, 12 steps = 60m
    group['occ_lag_15'] = group['occupancy_pct'].shift(3)
    group['occ_lag_30'] = group['occupancy_pct'].shift(6)
    group['occ_lag_60'] = group['occupancy_pct'].shift(12)
    
    # Target: 60 minutes into the future (12 steps ahead)
    group['future_occ_pct'] = group['occupancy_pct'].shift(-12)
    return group

print("Constructing segment-level lag features and 60-minute future target...")
df_fe = df_clean.groupby('segmentid', group_keys=False).apply(create_lags_and_target)

# Drop boundary rows that do not have full 60m history or 60m future target
df_fe = df_fe.dropna(subset=['occ_lag_15', 'occ_lag_30', 'occ_lag_60', 'future_occ_pct']).copy()

# Add weekend indicator
df_fe['is_weekend'] = (df_fe['dayofweek'] >= 5).astype(int)

# Target Discretization: 0 = Available, 1 = Nearly Full, 2 = Full
def encode_target(pct):
    if pct < 50.0:
        return 0  # Available
    elif pct < 80.0:
        return 1  # Nearly Full
    else:
        return 2  # Full

df_fe['target'] = df_fe['future_occ_pct'].apply(encode_target).astype('int8')

# Current baseline class (Persistence predictor)
df_fe['current_class_code'] = df_fe['occupancy_pct'].apply(encode_target).astype('int8')

# Keep one reading per segment per hour to prevent adjacent-row autocorrelation leakage
df_model = df_fe[df_fe['timestamp'].dt.minute == 4].copy()
if len(df_model) < 5000:
    # If minute==4 filter is too sparse for small samples, keep minute % 15 == 0
    df_model = df_fe[df_fe['timestamp'].dt.minute % 15 == 0].copy()

print(f"\nFinal Aligned Modeling Dataset: {len(df_model):,} rows")
print("\nTarget Class Distribution:")
target_map = {0: 'Available', 1: 'Nearly Full', 2: 'Full'}
for code, count in df_model['target'].value_counts().items():
    print(f"Class {code} ({target_map[code]}): {count:,} ({count/len(df_model)*100:.2f}%)")

# Inspect aligned sample row for viva demonstration
sample_cols = ['timestamp', 'segmentid', 'capacity', 'occupancy_pct', 'occ_lag_15', 'occ_lag_30', 'occ_lag_60', 'future_occ_pct', 'target']
print("\nAligned Sample Verification (Features vs. Future Target):")
print(df_model[sample_cols].head(3))


## Cell 6: Chronological Train-Validation-Test Splitting
In time-series parking prediction, we **MUST NOT** use an ordinary random `train_test_split()`.
A random split causes future-data leakage because nearby readings in time share almost identical traffic conditions.

Instead, we perform a **strictly chronological split**:
- **Training Set:** Earlier observations (~70%)
- **Validation Set:** Middle observations (~15% for model selection)
- **Test Set:** Latest observations (~15% held-out test)


In [ ]:
# Sort strictly by timestamp
df_model = df_model.sort_values(by='timestamp').reset_index(drop=True)

FEATURE_COLS = [
    'capacity', 'occupied', 'occupancy_pct',
    'occ_lag_15', 'occ_lag_30', 'occ_lag_60',
    'hour', 'dayofweek', 'is_weekend'
]
TARGET_COL = 'target'

n_total = len(df_model)
train_end = int(0.70 * n_total)
val_end = int(0.85 * n_total)

df_train = df_model.iloc[:train_end].copy()
df_val = df_model.iloc[train_end:val_end].copy()
df_test = df_model.iloc[val_end:].copy()

print("--- Chronological Split Summary ---")
print(f"Train Set      : {len(df_train):,} rows ({df_train['timestamp'].min()} to {df_train['timestamp'].max()})")
print(f"Validation Set : {len(df_val):,} rows ({df_val['timestamp'].min()} to {df_val['timestamp'].max()})")
print(f"Test Set       : {len(df_test):,} rows ({df_test['timestamp'].min()} to {df_test['timestamp'].max()})")

X_train, y_train = df_train[FEATURE_COLS], df_train[TARGET_COL]
X_val, y_val = df_val[FEATURE_COLS], df_val[TARGET_COL]
X_test, y_test = df_test[FEATURE_COLS], df_test[TARGET_COL]

# Fast training subset for SVM & Colab memory limits (matching report: 40k general, 10k SVM)
train_sample_size = min(40000, len(X_train))
X_train_sub = X_train.sample(n=train_sample_size, random_state=42)
y_train_sub = y_train.loc[X_train_sub.index]

svm_sample_size = min(10000, len(X_train))
X_train_svm = X_train.sample(n=svm_sample_size, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

print(f"Fast Training Sample: {train_sample_size:,} rows (SVM sample: {svm_sample_size:,} rows)")


## Cell 7: Build All 8 Classification Models (Experiment 9 Syllabus)
We implement all 8 syllabus algorithms using scikit-learn:
1. **Logistic Regression** (Linear baseline with StandardScaler)
2. **Naive Bayes** (GaussianNB)
3. **Artificial Neural Network** (MLPClassifier with hidden layers 64, 32)
4. **Support Vector Machine** (SVC with RBF kernel and scaling)
5. **K-Nearest Neighbors** (KNeighborsClassifier with k=7 and scaling)
6. **Decision Tree** (DecisionTreeClassifier with max_depth=8)
7. **Random Forest** (RandomForestClassifier with 100 trees)
8. **AdaBoost** (AdaBoostClassifier with 100 estimators)

We also include the **Persistence Baseline** (`y_pred = current_class(t)`), representing "next hour status equals current status".


In [ ]:
# Define models dictionary
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    
    "Naive Bayes": Pipeline([
        ('clf', GaussianNB())
    ]),
    
    "Artificial Neural Net (MLP)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42))
    ]),
    
    "Support Vector Machine (RBF)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=42))
    ]),
    
    "K-Nearest Neighbors (KNN)": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
    
    "Decision Tree": Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42))
    ]),
    
    "Random Forest": Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    
    "AdaBoost": Pipeline([
        ('clf', AdaBoostClassifier(n_estimators=100, random_state=42))
    ])
}

# Train each model
fitted_models = {}
for name, model in models.items():
    print(f"Training {name}...", end=" ")
    if name == "Support Vector Machine (RBF)":
        model.fit(X_train_svm, y_train_svm)
    else:
        model.fit(X_train_sub, y_train_sub)
    fitted_models[name] = model
    print("Done!")

print("\nAll 8 classification models successfully trained!")


## Cell 8: Model Evaluation and Performance Analytics
In this cell, we evaluate all models on the identical **validation set**.
We calculate:
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-Score
- Confusion Matrix
- Evaluation against Persistence Baseline


In [ ]:
results = []

# 1. Evaluate Persistence Baseline
y_val_persistence = df_val['current_class_code']
acc_base = accuracy_score(y_val, y_val_persistence)
prec_base = precision_score(y_val, y_val_persistence, average='macro', zero_division=0)
rec_base = recall_score(y_val, y_val_persistence, average='macro', zero_division=0)
f1_base = f1_score(y_val, y_val_persistence, average='macro', zero_division=0)

results.append({
    "Model": "Baseline (Persistence)",
    "Accuracy": acc_base,
    "Macro Precision": prec_base,
    "Macro Recall": rec_base,
    "Macro F1": f1_base
})

# 2. Evaluate All 8 ML Classifiers
for name, model in fitted_models.items():
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_val, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro Precision": prec,
        "Macro Recall": rec,
        "Macro F1": f1
    })

# Format into comparison dataframe sorted by Macro F1
df_results = pd.DataFrame(results).sort_values(by="Macro F1", ascending=False).reset_index(drop=True)
print("\n" + "="*70)
print("EXPERIMENT 9 MODEL COMPARISON TABLE (VALIDATION SET)")
print("="*70)
print(df_results.to_string(index=True, float_format=lambda x: f"{x:.4f}"))

# Visualize Performance Comparison
fig, ax = plt.subplots(figsize=(12, 6))
df_plot = df_results.sort_values(by="Macro F1", ascending=True)
colors_bar = ['#ef4444' if 'Naive' in m else ('#3b82f6' if 'Forest' in m else '#94a3b8') for m in df_plot['Model']]
bars = ax.barh(df_plot['Model'], df_plot['Macro F1'], color=colors_bar, height=0.6)
ax.set_xlabel("Macro F1-Score", fontweight='bold')
ax.set_title("Validation Macro F1-Score Comparison Across 8 Classifiers + Baseline", fontweight='bold')
ax.set_xlim(0.3, 0.85)

# Add data value labels
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.01, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Classification Report & Confusion Matrix for Selected Model (Random Forest)
best_model_name = "Random Forest"
best_model = fitted_models[best_model_name]
y_val_rf = best_model.predict(X_val)

print(f"\n--- Detailed Classification Report: {best_model_name} ---")
print(classification_report(y_val, y_val_rf, target_names=['Available', 'Nearly Full', 'Full'], digits=3))

cm = confusion_matrix(y_val, y_val_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Available', 'Nearly Full', 'Full'], yticklabels=['Available', 'Nearly Full', 'Full'])
plt.title(f"Confusion Matrix: {best_model_name} (Validation Set)", fontweight='bold')
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()


## Cell 9: Hyperparameter Tuning
We perform hyperparameter tuning using a time-series aware split for:
1. **Decision Tree:** `max_depth`, `min_samples_split`, `min_samples_leaf`
2. **Random Forest:** `n_estimators`, `max_depth`, `min_samples_leaf`
3. **Support Vector Machine:** `C`, `gamma`


In [ ]:
print("--- Hyperparameter Tuning: Random Forest ---")
rf_param_grid = {
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [8, 12, 16],
    'clf__min_samples_leaf': [2, 5]
}

# 3-Fold Time Series Cross Validation (ensures past trains on future test)
tscv = TimeSeriesSplit(n_splits=3)

grid_rf = GridSearchCV(
    fitted_models["Random Forest"],
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1
)

# Tune on train sample
grid_rf.fit(X_train_sub, y_train_sub)
print("Best Random Forest Parameters:", grid_rf.best_params_)
print(f"Best CV Macro F1: {grid_rf.best_score_:.4f}")

# Compare tuned vs untuned on validation set
tuned_rf = grid_rf.best_estimator_
y_val_tuned = tuned_rf.predict(X_val)
f1_tuned = f1_score(y_val, y_val_tuned, average='macro')
print(f"Untuned Validation Macro F1: {df_results.loc[df_results['Model']=='Random Forest', 'Macro F1'].values[0]:.4f}")
print(f"Tuned Validation Macro F1  : {f1_tuned:.4f}")


## Cell 10: Final Model Selection and Saving Artifacts
In this cell, we evaluate our selected model (**Random Forest**) once on the **held-out chronological test set**.
Then, we save the trained pipeline, feature list, segment profile dictionary, and evaluation metadata using `joblib`.


In [ ]:
# Final evaluation on held-out test set
y_test_pred = tuned_rf.predict(X_test)
y_test_base = df_test['current_class_code']

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
test_rec = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)

base_test_acc = accuracy_score(y_test, y_test_base)
base_test_f1 = f1_score(y_test, y_test_base, average='macro', zero_division=0)

print("="*60)
print("FINAL HELD-OUT TEST EVALUATION (Evaluated Once)")
print("="*60)
print(f"Random Forest Test Accuracy       : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Random Forest Test Macro Precision: {test_prec:.4f}")
print(f"Random Forest Test Macro Recall   : {test_rec:.4f}")
print(f"Random Forest Test Macro F1-Score : {test_f1:.4f}")
print(f"Persistence Baseline Test Acc     : {base_test_acc:.4f}")
print(f"Persistence Baseline Test Macro F1: {base_test_f1:.4f}")
print("="*60)

# Save model and metadata into models/ folder
os.makedirs("models", exist_ok=True)
MODEL_SAVE_PATH = os.path.join("models", "parking_model.joblib")
META_SAVE_PATH = os.path.join("models", "model_metadata.joblib")

joblib.dump(tuned_rf, MODEL_SAVE_PATH)

metadata = {
    "model_name": "Random Forest Classifier (Tuned)",
    "feature_names": FEATURE_COLS,
    "classes": ["Available", "Nearly Full", "Full"],
    "metrics": {
        "accuracy": float(test_acc),
        "macro_precision": float(test_prec),
        "macro_recall": float(test_rec),
        "macro_f1": float(test_f1)
    },
    "trained_date": str(datetime.datetime.now())
}
joblib.dump(metadata, META_SAVE_PATH)

print(f"Trained model saved to: {MODEL_SAVE_PATH}")
print(f"Metadata saved to: {META_SAVE_PATH}")

# Colab file download code (uncomment in Google Colab to download artifacts)
# from google.colab import files
# files.download(MODEL_SAVE_PATH)
# files.download(META_SAVE_PATH)


## Cell 11: Create Streamlit Application File (`app.py`)
In Google Colab, you can execute this cell with the `%%writefile app.py` magic command
to write out the complete interactive dashboard.


In [ ]:
# In Colab, execute: %%writefile app.py
# (See app.py in repository for the full Streamlit UI implementation)
print("app.py is prepared in repository root. Ready for deployment!")


## Cell 12: Run Streamlit and Expose via Public Tunnel in Colab
In this cell, we start Streamlit in the background and expose it using a public tunnel
so that our college teacher ("mam") and evaluators can interact with the live dashboard from any browser.


In [ ]:
# [FOR GOOGLE COLAB EXECUTION]
# Step 1: Launch Streamlit in background on port 8501
# !nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

# Step 2: Expose port 8501 via localtunnel
# !npx localtunnel --port 8501
